In [1]:
# !pip install slices numba -q

In [2]:
# !pip install --force-reinstall 'tensorflow[and-cuda]'

In [3]:
# !pip install torch torchvision torchaudio --force-reinstall --index-url https://download.pytorch.org/whl/cu128

In [4]:
# !pip install --force-reinstall "numpy==1.26.4"

In [5]:
# !pip install "tensorflow==2.8.0"

In [6]:
# !pip uninstall tensorflow -y
# !pip install --force-reinstall "tensorflow-gpu[tensorflow_with_gpu]==2.8.0"

In [7]:
# !pip install --force-reinstall "keras==3.5.0"

In [9]:
!cd .. && ./apply_patch.sh && cd -

/root/Crystal-generator-using-SLICES/notebooks


In [10]:
import sys, os
# sys.path.insert(0, os.path.abspath('..'))
sys.path.append('..')

In [11]:
from pymatgen.core import Structure
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

def compare_structures(s1, s2, to_print=False):
    # s1 = original_structure
    # s2 = backend.SLICES2structure(slices_NdSiRu)[0]

    ltol=0.3
    stol=0.5
    angle_tol=5

    
    # 2. Compare Lattice parameters manually (optional)
    tol = ltol  # 1% tolerance
    for a, b in zip(s1.lattice.abc, s2.lattice.abc):
        if abs(a-b)/max(a,b) > tol:
            if to_print: print(f"Lattice lengths differ by more than {int(ltol * 100)}%:", s1.lattice.abc, s2.lattice.abc)
            return False
    for α, β in zip(s1.lattice.angles, s2.lattice.angles):
        if abs(α-β) > angle_tol:  # 1° tolerance
            if to_print: print("Lattice angles differ by more than {angle_tol}°:", s1.lattice.angles, s2.lattice.angles)
            return False

    # 3. Compare Space Groups
    # spg1 = SpacegroupAnalyzer(s1).get_space_group_symbol()
    # spg2 = SpacegroupAnalyzer(s2).get_space_group_symbol()
    # if spg1 != spg2:
    #     print(f"Different space groups: {spg1} vs {spg2}")
    # else:
    #     print(f"Same space group: {spg1}")

    # 4. Fuzzy structure matching (accounts for cell reduction, slight distortions,
    #    ordering of sites, and atomic‐type permutations)

    # print(s1.composition)
    # print(s2.composition)

    # print(s2.lattice)
    # print(len(s2.sites))
    # print('---------------------')
    # print(s1.lattice)
    # print(len(s1.sites))

    # if any(not site.is_ordered for site in s2.sites):
    #     print("Partial occupancy detected.")

    # import numpy as np
    # if np.any(np.isnan([c for site in s2.sites for c in site.frac_coords])):
    #     print("NaN in coordinates!")

    matcher = StructureMatcher(
        ltol=ltol,  # lattice length tolerance (5%)
        stol=stol,   # site position tolerance (Å)
        angle_tol=angle_tol,  # angle tolerance (°)
        primitive_cell=True,
        scale=False
    )

    # print(matcher.fit(s2, s2))
    # print('--------------')
    # print(matcher.fit(s1, s1))

    # return False

    are_fit = matcher.fit(s1.get_primitive_structure(), s2.get_primitive_structure())

    if to_print:
        if are_fit:
            print("Structures match (within tolerances) 🎉")
        else:
            print("Structures do not match.")
    return are_fit

In [12]:
import re

def get_materail_ids(file):
    rows = []
    pattern = r'^(?P<status>E|x)?\s*(?P<idx>\d+):\s*(?P<material>[a-zA-Z]+-\d+)\s*---\s*(?P<time>[\d\.eE+-]+)'

    for line in open(file).read().strip().split('\n'):
        m = re.match(pattern, line)
        if m:
            rows.append(m.group('material'))
    return rows

In [13]:
from src.slices_pipeline.encoder import *
from src.slices_pipeline.loader import *
from slices.core import SLICES
from pymatgen.core.structure import Structure
from multiprocessing import Process, Queue
from pymatgen.io.cif import CifParser
from io import StringIO
from tqdm import tqdm
import time
import os

def job(rows, queue, worker_id):
    # Setup: e.g., load a model, establish a connection, etc.
    # print(f"Worker {worker_id} setup complete")

    # backend = SLICES(relax_model="chgnet", steps=100)
    backend = SLICES()

    for idx, row in rows.iterrows():
        # Simulate processing (replace with your logic)
        # result = row['value'] ** 2

        start_time = time.time()
        material_id = row.material_id
        try:
            # original_structure = Structure.from_str(cif, fmt="cif", primitive=True)
            cif = row.cif
            original_structure = CifParser(StringIO(cif)).parse_structures()[0]
            # print(i, df.iloc[i].material_id)
            slices_NdSiRu = backend.structure2SLICES(original_structure)
            reconstructed_structure, final_energy_per_atom = backend.SLICES2structure(slices_NdSiRu)
            is_okay = compare_structures(original_structure, reconstructed_structure)
            # if is_okay: correct += 1
            end_time = time.time()
            # data.append(f"{' ' if is_okay else 'x'} {i}: {df.iloc[i]['material_id']} --- {end_time - start_time}")
            queue.put((worker_id, f"{' ' if is_okay else 'x'} {idx}: {material_id} --- {end_time - start_time}"))
        except Exception:
            # error += 1
            end_time = time.time()
            # data.append(f"E {i}: {df.iloc[i]['material_id']} --- {end_time - start_time}")
            queue.put((worker_id, f"E {idx}: {material_id} --- {end_time - start_time}"))

        # queue.put((worker_id, idx))  # Report subtask done

    queue.put((worker_id, None))  # Signal this worker is done

def run_split_parallel(split, start_from=0, append_previous_result=True, jobs=2):
    if split is None:
        df = pd.concat([read_split_df("train"), read_split_df("test"), read_split_df("val")])
        split = "full"
    else:
        df = read_split_df(split)
    correct = 0
    overall = 0
    error = 0
    initial_size = df.shape[0]
    file_to_save = f"results-{split}.txt"
    if (append_previous_result or start_from > 0) and os.path.exists(file_to_save):
        with open(file_to_save, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if len(lines) > 0:
                if not lines[-1]:
                    lines = lines[:-1]
                num_lines = len(lines)
                correct = sum(1 for l in lines if l[0] == ' ')
                overall = num_lines
                error = sum(1 for l in lines if l[0] == 'E')
                start_from = num_lines
                to_remove_ids = get_materail_ids(file_to_save)
                df = df[~df['material_id'].isin(to_remove_ids)]
    data = []
    # backend = SLICES(relax_model="chgnet", steps=100)
    # backend = SLICES()
    
    batch_size = df.shape[0] // jobs
    queue = Queue()

    processes = []
    for i in range(jobs):
        start = i * batch_size
        end = None if i == jobs-1 else (i+1) * batch_size
        batch = df.iloc[start:end]
        p = Process(target=job, args=(batch, queue, i))
        processes.append(p)
        p.start()

    i = start_from
    finished_workers = 0
    with tqdm(total=initial_size, initial=start_from, smoothing=0.1) as pbar:
        while finished_workers < jobs:
            worker_id, string = queue.get()
            if string is not None:
                if string[0] == 'E':
                    error += 1
                elif string[0] == ' ':
                    correct += 1

                overall += 1
                data.append(string)

                pbar.set_postfix(correct=f"{correct}/{overall}", accuracy=f"{correct/overall:.2%}", error=f"{error}")
                pbar.update(1)
                i += 1

                if i % 10 == 9:
                    with open(file_to_save, "a", encoding="utf-8") as f:
                        f.writelines(line + "\n" for line in data)
                    data = []
            else:
                finished_workers += 1
        with open(file_to_save, "a", encoding="utf-8") as f:
            f.writelines(line + "\n" for line in data)
    for p in processes:
        p.join()



    # with tqdm(total=df.shape[0], initial=start_from, smoothing=0.9) as pbar:
    #     for i, cif in enumerate(df.cif.iloc[start_from:], start=start_from):
    #         start_time = time.time()
    #         try:
    #             # original_structure = Structure.from_str(cif, fmt="cif", primitive=True)
    #             original_structure = CifParser(StringIO(cif)).parse_structures()[0]
    #             # print(i, df.iloc[i].material_id)
    #             slices_NdSiRu = backend.structure2SLICES(original_structure)
    #             reconstructed_structure, final_energy_per_atom = backend.SLICES2structure(slices_NdSiRu)
    #             is_okay = compare_structures(original_structure, reconstructed_structure)
    #             if is_okay: correct += 1
    #             end_time = time.time()
    #             data.append(f"{' ' if is_okay else 'x'} {i}: {df.iloc[i]['material_id']} --- {end_time - start_time}")
    #         except Exception:
    #             error += 1
    #             end_time = time.time()
    #             data.append(f"E {i}: {df.iloc[i]['material_id']} --- {end_time - start_time}")
    #         finally:
    #             overall += 1
    #             pbar.set_postfix(correct=f"{correct}/{overall}", accuracy=f"{correct/overall:.2%}", error=f"{error}")
    #             pbar.update(1)
    #             if i % 10 == 9:
    #                 with open(file_to_save, "a", encoding="utf-8") as f:
    #                     f.writelines(line + "\n" for line in data)
    #                 data = []
    # with open(file_to_save, "a", encoding="utf-8") as f:
    #     f.writelines(line + "\n" for line in data)

E0000 00:00:1748539309.471214   21597 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748539309.481104   21597 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748539309.506046   21597 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748539309.506071   21597 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748539309.506073   21597 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748539309.506075   21597 computation_placer.cc:177] computation placer already registered. Please check linka

Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


In [14]:
# !pip3 install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [23]:
# run_split_parallel("test", jobs=4)
run_split_parallel(None, jobs=1)

  0%|          | 169/45229 [00:00<?, ?it/s]

Process Process-1:
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_21597/131281227.py", line 17, in job
    backend = SLICES()
  File "/root/Crystal-generator-using-SLICES/.venv/lib/python3.10/site-packages/slices/core.py", line 135, in __init__
    self.relaxer = Relaxer(optimizer=optimizer)
  File "/root/Crystal-generator-using-SLICES/.venv/lib/python3.10/site-packages/m3gnet/models/_dynamics.py", line 122, in __init__
    potential = Potential(M3GNet.load())
  File "/root/Crystal-generator-using-SLICES/.venv/lib/python3.10/site-packages/m3gnet/models/_m3gnet.py", line 363, in load
    return cls.load(os.path.join(CWD, model_name))
  File "/root/Crystal-generator-using-SLICES/.venv/lib/python3.10/site-packages/m3gnet/models/_m3gnet.py", line 367, in load
    ret

KeyboardInterrupt: 

In [ ]:
# run_split_parallel("test", jobs=4)

 16%|█▌        | 1249/7797 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1186: UserWarning: The default value of primitive was changed from True to False in https://github.com/materialsproject/pymatgen/pull/3419. CifParser now returns the cell in the CIF file as is. If you want the primitive cell, please set primitive=True explicitly.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1186: UserWarning: The default value of primitive was changed from True to False in https://github.com/materialsproject/pymatgen/pull/3419. CifParser now returns the cell in the CIF file as is. If you want the primitive cell, please set primitive=True explicitly.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1186: UserWarning: The default value of primitive was changed from True to False in https://github.com/materialsproject/pymatgen/pull/3419. CifParser now returns the cell in the CIF file as is. If you want the primitiv

ERROR - could not find a linearly independent cocycle basis!


 18%|█▊        | 1377/7797 [03:54<4:55:21,  2.76s/it, accuracy=84.46%, correct=1163/1377, error=133]/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1219: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
 18%|█▊        | 1380/7797 [03:58<1:16:35,  1.40it/s, accuracy=84.49%, correct=1166/1380, error=133]/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1219: UserWarning: Issues encountered while parsing CIF: 10 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
ERROR:root:Could not obtain the lattice basis from the cycle vectors!


ERROR - could not find a linearly independent cocycle basis!


 18%|█▊        | 1382/7797 [04:00<2:21:15,  1.32s/it, accuracy=84.52%, correct=1168/1382, error=133]/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1219: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
 18%|█▊        | 1396/7797 [04:25<3:09:29,  1.78s/it, accuracy=84.53%, correct=1180/1396, error=135]/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1219: UserWarning: Issues encountered while parsing CIF: 6 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
 18%|█▊        | 1417/7797 [05:00<3:15:44,  1.84s/it, accuracy=84.69%, correct=1200/1417, error=135]ERROR:root:Could not obtain the lattice basis from the cycle vectors!


...

ERROR - could not find a linearly independent cocycle basis!


 21%|██▏       | 1668/7797 [13:25<3:16:27,  1.92s/it, accuracy=84.83%, correct=1415/1668, error=156]Process Process-4:
Process Process-3:
Traceback (most recent call last):
Process Process-1:
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
Process Process-2:
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_50523/2625565288.py", line 31, in job
    reconstructed_structure, final_energy_per_atom = backend.SLICES2structure(slices_NdSiRu)
  File "/tmp/ipykernel_50523/2625565288.py", line 31, in job
    reconstructed_structure, final_energy_per_atom = backend.SLICES2structure(slices_NdSiRu)
  File "/usr/l

KeyboardInterrupt: 

  File "/usr/local/lib/python3.10/dist-packages/slices/core.py", line 1675, in func
    metric_tensor, cocycle_rep = self.convert_params(x, ndim, int(order - 1),lattice_type,metric_tensor_std)
  File "/usr/local/lib/python3.10/dist-packages/numpy/lib/twodim_base.py", line 1115, in triu_indices
    return tuple(broadcast_to(inds, tri_.shape)[tri_]
  File "/usr/local/lib/python3.10/dist-packages/ase/optimize/optimize.py", line 269, in run
    return Dynamics.run(self)
  File "/usr/local/lib/python3.10/dist-packages/slices/core.py", line 1520, in convert_params
    g = np.triu_indices(ndim, 1)
  File "/usr/local/lib/python3.10/dist-packages/numpy/lib/twodim_base.py", line 1115, in <genexpr>
    return tuple(broadcast_to(inds, tri_.shape)[tri_]
  File "/usr/local/lib/python3.10/dist-packages/ase/optimize/optimize.py", line 156, in run
    for converged in Dynamics.irun(self):
  File "/usr/local/lib/python3.10/dist-packages/numpy/lib/twodim_base.py", line 1113, in triu_indices
    tri_ = ~t

  File "/usr/local/lib/python3.10/dist-packages/numpy/lib/stride_tricks.py", line 413, in broadcast_to
    return _broadcast_to(array, shape, subok=subok, readonly=True)
  File "/usr/local/lib/python3.10/dist-packages/ase/optimize/optimize.py", line 135, in irun
    self.step()
  File "/usr/local/lib/python3.10/dist-packages/numpy/lib/twodim_base.py", line 414, in tri
    m = greater_equal.outer(arange(N, dtype=_min_int(0, N)),
  File "/usr/local/lib/python3.10/dist-packages/numpy/lib/stride_tricks.py", line 349, in _broadcast_to
    it = np.nditer(
KeyboardInterrupt
  File "/usr/local/lib/python3.10/dist-packages/ase/optimize/bfgs.py", line 87, in step
    r = atoms.get_positions()
KeyboardInterrupt
  File "/usr/local/lib/python3.10/dist-packages/ase/constraints.py", line 2604, in get_positions
    pos[natoms:] = logm(self.deform_grad())
  File "/usr/local/lib/python3.10/dist-packages/scipy/linalg/_matfuncs.py", line 203, in logm
    F = scipy.linalg._matfuncs_inv_ssq._logm(A)
  File 

In [ ]:
# run_split_parallel("test")

  0%|          | 0/9046 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1186: UserWarning: The default value of primitive was changed from True to False in https://github.com/materialsproject/pymatgen/pull/3419. CifParser now returns the cell in the CIF file as is. If you want the primitive cell, please set primitive=True explicitly.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1186: UserWarning: The default value of primitive was changed from True to False in https://github.com/materialsproject/pymatgen/pull/3419. CifParser now returns the cell in the CIF file as is. If you want the primitive cell, please set primitive=True explicitly.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1219: UserWarning: Issues encountered while parsing CIF: 16 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.w

ERROR - could not find a linearly independent cocycle basis!


  0%|          | 1/9046 [00:01<2:49:53,  1.13s/it, accuracy=0.00%, correct=0/1, error=1]

[Errno 2] No such file or directory: '/dev/shm/tmpzofmdxer/gfnff_lists.json'


  0%|          | 8/9046 [00:09<1:45:16,  1.43it/s, accuracy=62.50%, correct=5/8, error=2]/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1219: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
ERROR:root:Could not obtain the lattice basis from the cycle vectors!


...

[Errno 2] No such file or directory: '/dev/shm/tmpu39pwb2b/gfnff_lists.json'


 13%|█▎        | 1154/9046 [31:08<1:22:12,  1.60it/s, accuracy=84.40%, correct=974/1154, error=114]

[Errno 2] No such file or directory: '/dev/shm/tmpyztk0i61/gfnff_lists.json'


 14%|█▍        | 1253/9046 [33:55<3:30:58,  1.62s/it, accuracy=84.28%, correct=1056/1253, error=123] Process Process-7:
Process Process-8:
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_18998/2447446525.py", line 31, in job
    reconstructed_structure, final_energy_per_atom = backend.SLICES2structure(slices_NdSiRu)
  File "/usr/local/lib/python3.10/dist-packages/slices/core.py", line 2181, in SLICES2structure
    structures,final_energy_per_atom = self.to_structures()
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/slices/core.py", line 2113, in to_structures
    x=fmin_l_bfgs_b(self.func, x, fprime=None, args= \
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/local/li

KeyboardInterrupt: 